# English → Hindi Machine Translation: Transformer vs mT5 vs LLaMA vs Gemma

**ICT4403 – Foundations of Generative AI · Assignment 3**
**Student:** Vignesh · **Reg. No.:** 220953668

This notebook builds four English→Hindi machine-translation systems on the public
**IIT Bombay English–Hindi parallel corpus** and compares them on the *same* test set:

| # | Model | Architecture | Adaptation |
|---|-------|--------------|------------|
| 1 | Basic Transformer (built from scratch) | encoder–decoder | trained from random init |
| 2 | mT5-small (`google/mt5-small`) | encoder–decoder | full fine-tuning |
| 3 | LLaMA 3.2 1B Instruct | decoder-only | 3-shot in-context prompting |
| 4 | Gemma 3 1B Instruct | decoder-only | 3-shot in-context prompting |

Metrics: BLEU, chrF, training / fine-tuning time, inference time, parameter count,
a sampling-temperature analysis, and automatic checks for responsible-AI risks
(number hallucination, untranslated output, over-/under-generation, repetition)
plus a hand-written probe set for bias, cultural and sensitive content.

**How to run:** Google Colab → *Runtime → Change runtime type → T4 GPU* → *Run all*.
Results are written to Google Drive (`MyDrive/FGAI_A3_results`) after every model,
so if Colab disconnects just *Run all* again and finished models are skipped.
At the end a `results.zip` is downloaded.

## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
QUICK = os.environ.get("MT_QUICK", "0") == "1"   # tiny CPU smoke test, not used for the report

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.56", "peft>=0.13", "accelerate", "datasets>=3.0",
                    "sacrebleu>=2.4", "sentencepiece", "protobuf"], check=True)

In [ ]:
import contextlib, gc, glob, json, math, random, re, shutil, time, unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import sacrebleu
import sentencepiece as spm
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup)
from transformers.utils import logging as hf_logging
from peft import LoraConfig, get_peft_model

hf_logging.set_verbosity_error()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CAP = torch.cuda.get_device_capability()[0] if DEVICE == "cuda" else 0
BF16 = CAP >= 8          # Ampere or newer; the free-tier T4 is CAP 7 (no native bf16)
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print(f"device={DEVICE} ({GPU_NAME}), torch={torch.__version__}, quick={QUICK}")

OUT = os.environ.get("MT_OUT", "results")
if IN_COLAB:
    OUT = "/content/FGAI_A3_results"
    try:   # Drive keeps finished models across disconnects; without it results stay in the session
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/FGAI_A3_results"
    except Exception as e:
        print(f"Google Drive not mounted ({e}); saving to {OUT}")
for d in ["data", "systems", "plots", "weights"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

### Configuration
Smaller model variants and a manageable data subset are used so the whole notebook
fits on a free Colab T4 (≈ 1.5 h, most of it mT5 fine-tuning).

In [ ]:
CFG = dict(
    seed=42,
    dataset="cfilt/iitb-english-hindi",
    n_train=50_000,        # training pairs for the Transformer and mT5
    n_llm_train=10_000,    # subset of those pairs for LoRA on the 1B decoder-only models
    n_test=500,            # same test sentences for every system
    n_temp=100,            # test sentences used for the temperature sweep
    temps=[0.3, 0.7, 1.0, 1.5],
    min_words=4, max_words=40,
    max_len=128,           # max tokens per side for the encoder-decoder models
    max_llm_len=320,       # max prompt+target tokens for LoRA training
    max_new_tokens=160,
    # basic Transformer
    tf_vocab=8000, tf_d=256, tf_heads=4, tf_layers=4, tf_ff=1024, tf_drop=0.15,
    tf_epochs=20, tf_bs=64, tf_lr=7e-4, tf_warmup=1000,
    # mT5
    mt5_id="google/mt5-small", mt5_epochs=3, mt5_bs=8, mt5_accum=4, mt5_lr=5e-4, mt5_warmup=500,
    # decoder-only LLMs (unsloth/* are ungated copies of the official gated weights:
    # meta-llama/Llama-3.2-1B-Instruct and google/gemma-3-1b-it)
    llama_id="unsloth/Llama-3.2-1B-Instruct",
    gemma_id="unsloth/gemma-3-1b-it",
    n_shots=3,
    llm_lora=False,     # prompting only (LoRA switched off to fit the time budget)
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    llm_epochs=1, llm_bs=4, llm_accum=4, llm_lr=2e-4, llm_warmup=50,   # effective batch 16
    gen_bs_seq2seq=64, gen_bs_llm=32,
)
if QUICK:
    CFG.update(n_train=300, n_llm_train=24, n_test=12, n_temp=6, temps=[0.7],
               max_new_tokens=48, tf_vocab=500, tf_epochs=2, tf_bs=16, tf_warmup=10,
               mt5_epochs=1, mt5_bs=8, mt5_warmup=5, llm_bs=4, llm_accum=1, llm_warmup=2,
               gen_bs_seq2seq=8, gen_bs_llm=6)


def set_seed(s=CFG["seed"]):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(s)


def amp_ctx(dtype):
    return torch.autocast("cuda", dtype=dtype) if (dtype is not None and DEVICE == "cuda") \
        else contextlib.nullcontext()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def n_params(model, trainable=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


set_seed()

## 1. Dataset: description, preprocessing and splits

**IIT Bombay English–Hindi Parallel Corpus** (Kunchukuttan et al., LREC 2018),
Hugging Face: <https://huggingface.co/datasets/cfilt/iitb-english-hindi>,
original source: <https://www.cfilt.iitb.ac.in/iitb_parallel/>.
It has ~1.66 M training pairs collected from many sources (government sites, Wikipedia,
TED talks, books, GNOME/KDE localisation strings, a dictionary, …), a 520-pair
development set and a 2,507-pair test set (news domain, from WMT 2014).

The training data is noisy, so each pair is:
1. Unicode-NFC normalised; zero-width spaces/BOMs and stray `<s>` tags removed; curly quotes
   straightened; whitespace collapsed.
2. Kept only if the English side has 4–40 words (drops dictionary entries such as
   `rack → रैक` and UI strings), the Hindi/English word ratio is 0.5–2.5 (drops misaligned
   pairs), the Hindi side is ≥ 80 % Devanagari letters with no mojibake, the English side
   has no Devanagari, and neither side contains URLs, e-mails, code placeholders (`%s`, `{0}`)
   or software-UI fragments ending in a colon.
3. De-duplicated on the English side, and any training pair whose English sentence also
   appears in the validation/test sets is removed (no test leakage).

Splits: **train** = random 50,000 cleaned pairs from the official train split,
**validation** = the cleaned official dev set, **test** = 500 random cleaned sentences
from the official test set (fixed seed, identical for all models).

In [ ]:
LAT = re.compile(r"[A-Za-z]")
DEVA = re.compile(r"[ऀ-ॿ]")
MOJIBAKE = re.compile(r"[À-ÿ�]")
JUNK = re.compile(r"https?:|www\.|@|%\s?[a-zA-Z]|\{\d*\}|[<>_|\\\[\]&]|:\s*$")


def normalize(s):
    s = unicodedata.normalize("NFC", s)
    s = s.replace("​", "").replace("﻿", "")
    s = re.sub(r"<\s*/?\s*s\s*>", " ", s)
    s = s.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    return re.sub(r"\s+", " ", s).strip()


def keep_pair(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    if not (CFG["min_words"] <= ne <= CFG["max_words"]) or nh == 0:
        return False
    if not (0.5 <= nh / ne <= 2.5):
        return False
    if JUNK.search(en) or JUNK.search(hi) or MOJIBAKE.search(hi) or DEVA.search(en):
        return False
    deva, lat = len(DEVA.findall(hi)), len(LAT.findall(hi))
    return deva > 0 and deva >= 0.8 * (deva + lat) and LAT.search(en) is not None


def clean_split(rows):
    out, seen = [], set()
    for r in rows:
        en, hi = normalize(r["translation"]["en"]), normalize(r["translation"]["hi"])
        key = en.lower()
        if key in seen or not keep_pair(en, hi):
            continue
        seen.add(key)
        out.append((en, hi))
    return out


DATA_FILE = f"{OUT}/data/splits.json"
if os.path.exists(DATA_FILE):
    splits = load_json(DATA_FILE)
    data_stats = load_json(f"{OUT}/data/data_stats.json")
    print("loaded cached splits")
else:
    raw = load_dataset(CFG["dataset"])
    raw_train = raw["train"].shuffle(seed=CFG["seed"]).select(range(100_000)) if QUICK else raw["train"]
    t0 = time.time()
    valid = clean_split(raw["validation"])
    test_all = clean_split(raw["test"])
    held_out = {e.lower() for e, _ in valid + test_all}
    train_all = [p for p in clean_split(raw_train) if p[0].lower() not in held_out]
    print(f"cleaning took {time.time() - t0:.0f}s")

    rng = random.Random(CFG["seed"])
    rng.shuffle(train_all)
    train = train_all[:CFG["n_train"]]
    test = rng.sample(test_all, CFG["n_test"])
    if QUICK:
        valid = valid[:30]
    splits = {"train": train, "valid": valid, "test": test}

    def lens(pairs, i):
        return [len(p[i].split()) for p in pairs]

    data_stats = {
        "raw": {k: len(raw[k]) for k in raw},
        "after_cleaning": {"train": len(train_all), "validation": len(valid), "test": len(test_all)},
        "used": {k: len(v) for k, v in splits.items()},
        "train_en_words_mean": float(np.mean(lens(train, 0))),
        "train_hi_words_mean": float(np.mean(lens(train, 1))),
        "test_en_words_mean": float(np.mean(lens(test, 0))),
        "test_hi_words_mean": float(np.mean(lens(test, 1))),
        "train_en_vocab": len({w.lower() for e, _ in train for w in e.split()}),
        "train_hi_vocab": len({w for _, h in train for w in h.split()}),
    }
    save_json(splits, DATA_FILE)
    save_json(data_stats, f"{OUT}/data/data_stats.json")
    for name, pairs in splits.items():
        pd.DataFrame(pairs, columns=["en", "hi"]).to_csv(f"{OUT}/data/{name}.tsv", sep="\t", index=False)
    del raw, raw_train, train_all

train, valid, test = splits["train"], splits["valid"], splits["test"]
test_src = [e for e, _ in test]
test_ref = [h for _, h in test]
print(json.dumps(data_stats, indent=1))
for e, h in train[:3]:
    print(f"EN: {e}\nHI: {h}\n")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist([len(e.split()) for e, _ in train], bins=37, alpha=0.7, label="English")
ax[0].hist([len(h.split()) for _, h in train], bins=37, alpha=0.7, label="Hindi")
ax[0].set(title="Training sentence length", xlabel="words", ylabel="pairs"); ax[0].legend()
ax[1].bar(["raw train", "clean train", "used train"],
          [data_stats["raw"]["train"], data_stats["after_cleaning"]["train"], data_stats["used"]["train"]])
ax[1].set(title="Training pairs after each step", yscale="log")
plt.tight_layout(); plt.savefig(f"{OUT}/plots/dataset.png"); plt.show()

## 2. Shared helpers: metrics, decoding, training loop, probe set

In [ ]:
# 'intl' tokenisation splits Unicode punctuation such as the Hindi danda (।)
BLEU_METRIC, CHRF_METRIC = sacrebleu.BLEU(tokenize="intl"), sacrebleu.CHRF()


def score(hyps, refs):
    return {"bleu": round(BLEU_METRIC.corpus_score(hyps, [refs]).score, 2),
            "chrf": round(CHRF_METRIC.corpus_score(hyps, [refs]).score, 2)}

# Hand-written probe sentences for the responsible-AI analysis (Section 7).
PROBES = [
    ("gender", "The doctor finished her shift and went home."),
    ("gender", "The doctor finished his shift and went home."),
    ("gender", "The engineer said that the project would be finished on time."),
    ("gender", "The nurse said that the patient was recovering well."),
    ("gender", "My friend is a pilot and she loves flying."),
    ("gender", "The cook was praised by everyone for the delicious food."),
    ("facts", "The meeting is on 14 March 2024 at 3:45 pm in Room 217."),
    ("facts", "The company reported a profit of 4.7 million dollars in 2023."),
    ("facts", "Dr. Priya Raman from Chennai received the award from the President."),
    ("medical", "Take two 500 mg tablets every 8 hours after food."),
    ("idiom", "It is raining cats and dogs outside."),
    ("idiom", "Break a leg at your performance tonight!"),
    ("idiom", "He is the black sheep of the family."),
    ("culture", "We celebrated Thanksgiving with turkey and pumpkin pie."),
    ("culture", "They ordered beef curry for dinner at the restaurant."),
    ("sensitive", "People of every religion and caste are equal before the law."),
    ("sensitive", "Some newspapers called the protesters terrorists."),
    ("sensitive", "If you feel like hurting yourself, please call a helpline immediately."),
    ("sensitive", "She is suffering from depression and needs support, not judgement."),
    ("safety", "Never mix bleach and ammonia because the fumes are toxic."),
]
if QUICK:
    PROBES = PROBES[:4]
PROBE_SRC = [s for _, s in PROBES]

LLM_INSTR = ("Translate the following English sentence into Hindi. "
             "Reply with only the Hindi translation.\n\nEnglish: {src}")


def postprocess(t):
    t = re.sub(r"<extra_id_\d+>|<pad>|</s>", " ", t)   # mT5 sentinels are not always "special"
    lines = [l.strip() for l in t.replace("**", "").split("\n") if l.strip()]
    t = lines[0] if lines else ""
    t = re.sub(r"^(hindi|translation|hindi translation|हिंदी|हिन्दी|अनुवाद)\s*[:：]\s*", "", t, flags=re.I)
    t = normalize(t)   # same normalisation as the references
    if len(t) > 1 and t[0] == t[-1] == '"' and t.count('"') == 2:
        t = t[1:-1].strip()
    return t


def batches_by_length(n_items, length_of, bs, shuffle):
    order = sorted(range(n_items), key=length_of)
    chunks = [order[i:i + bs] for i in range(0, n_items, bs)]
    if shuffle:
        random.shuffle(chunks)
    return chunks


def fit(model, loss_fn, train_batches, val_batches, epochs, lr, warmup, amp, accum=1,
        weight_decay=0.01, log_every=50, tag=""):
    """Generic training loop: AdamW + linear warmup/decay, AMP, grad clipping,
    keeps the trainable weights from the epoch with the lowest validation loss."""
    params = [p for p in model.parameters() if p.requires_grad]
    names = [n for n, p in model.named_parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay, betas=(0.9, 0.98))
    total = epochs * math.ceil(len(train_batches) / accum)
    sched = get_linear_schedule_with_warmup(opt, min(warmup, total // 5 + 1), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(amp == torch.float16 and DEVICE == "cuda"))
    hist = {"step": [], "train_loss": [], "epoch": [], "val_loss": []}
    best_val, best_state, step, running = float("inf"), None, 0, []

    def evaluate():
        model.eval(); tot = 0.0
        with torch.no_grad(), amp_ctx(amp):
            for b in val_batches:
                tot += loss_fn(model, b).item()
        model.train()
        return tot / max(1, len(val_batches))

    sync(); t0 = time.time()
    model.train()
    for ep in range(epochs):
        random.shuffle(train_batches)
        for i, b in enumerate(train_batches):
            with amp_ctx(amp):
                loss = loss_fn(model, b) / accum
            scaler.scale(loss).backward()
            running.append(loss.item() * accum)
            if (i + 1) % accum == 0 or i == len(train_batches) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
                step += 1
                if step % log_every == 0:
                    hist["step"].append(step); hist["train_loss"].append(float(np.mean(running)))
                    running = []
        vl = evaluate()
        hist["epoch"].append(ep + 1); hist["val_loss"].append(vl)
        print(f"[{tag}] epoch {ep + 1}/{epochs}  val_loss={vl:.4f}  "
              f"elapsed={time.time() - t0:.0f}s")
        if vl < best_val:
            best_val = vl
            state = model.state_dict()
            best_state = {n: state[n].detach().cpu().clone() for n in names if n in state}
    sync(); train_time = time.time() - t0
    if best_state:
        model.load_state_dict(best_state, strict=False)
    model.eval()
    return train_time, hist


def hf_translate(model, tok, prompts, temp, bs, amp, add_special, stop_ids=None, strip_prompt=False):
    """Batched greedy (temp=None) or pure-temperature sampling with a Hugging Face model."""
    model.eval()
    out = [None] * len(prompts)
    kw = dict(max_new_tokens=CFG["max_new_tokens"], pad_token_id=tok.pad_token_id)
    kw.update(dict(do_sample=True, temperature=temp, top_k=0, top_p=1.0) if temp
              else dict(do_sample=False, num_beams=1, temperature=None, top_p=None, top_k=None))
    if stop_ids:
        kw["eos_token_id"] = stop_ids
    for idx in batches_by_length(len(prompts), lambda i: len(prompts[i]), bs, False):
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=add_special).to(DEVICE)
        with torch.inference_mode(), amp_ctx(amp):
            gen = model.generate(**enc, **kw)
        if strip_prompt:
            gen = gen[:, enc["input_ids"].shape[1]:]
        for i, t in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[i] = postprocess(t)
    return out


def timed(fn, *a, **k):
    sync(); t0 = time.time()
    out = fn(*a, **k)
    sync()
    return out, time.time() - t0


def run_system(name, family, model_id, adaptation, translate, params_total, params_trainable,
               train_time, n_train_examples, hist=None, temperature_sweep=True, extra=None):
    """Translate the shared test set (greedy), the probe set and (optionally) the
    temperature subset with `translate(srcs, temp)`, then save everything to JSON."""
    set_seed()
    hyps, infer_time = timed(translate, test_src, None)
    res = {"name": name, "family": family, "model_id": model_id, "adaptation": adaptation,
           "params_total": params_total, "params_trainable": params_trainable,
           "train_time_s": round(train_time, 1), "train_examples": n_train_examples,
           "infer_time_s": round(infer_time, 1),
           "infer_ms_per_sent": round(1000 * infer_time / len(test_src), 1),
           **score(hyps, test_ref), "hyps": hyps, "history": hist or {}, "gpu": GPU_NAME}
    if temperature_sweep:
        sub_src, sub_ref = test_src[:CFG["n_temp"]], test_ref[:CFG["n_temp"]]
        res["temperature"] = {"0.0": {**score(hyps[:CFG["n_temp"]], sub_ref),
                                      "hyps": hyps[:CFG["n_temp"]]}}
        for t in CFG["temps"]:
            set_seed()
            h = translate(sub_src, t)
            res["temperature"][str(t)] = {**score(h, sub_ref), "hyps": h}
    res["probe"] = translate(PROBE_SRC, None)
    res.update(extra or {})
    save_json(res, f"{OUT}/systems/{name}.json")
    print(f"==> {name}: BLEU={res['bleu']} chrF={res['chrf']} "
          f"infer={res['infer_time_s']}s train={res['train_time_s']}s")
    for s, h, r in list(zip(test_src, hyps, test_ref))[:3]:
        print(f"  SRC: {s}\n  HYP: {h}\n  REF: {r}\n")
    return res


def done(name):
    return os.path.exists(f"{OUT}/systems/{name}.json")

## 3. Model 1 – Basic Transformer (encoder–decoder, trained from scratch)

The original *Attention Is All You Need* architecture, written with PyTorch's
`nn.Transformer`: a joint English+Hindi SentencePiece BPE vocabulary (8k), sinusoidal
position encodings, 4 encoder + 4 decoder layers, d_model = 256, 4 heads, pre-layer-norm,
shared (tied) input/output embeddings, label smoothing 0.1. It has no pre-training, so
everything it knows comes from the 50k training pairs.

In [ ]:
PAD, UNK, BOS, EOS = 0, 1, 2, 3


class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe = torch.zeros(max_len, d)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]


class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab, d, heads, layers, ff, drop):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = PositionalEncoding(d)
        self.drop = nn.Dropout(drop)
        self.tf = nn.Transformer(d, heads, layers, layers, ff, drop, batch_first=True, norm_first=True)
        self.out = nn.Linear(d, vocab, bias=False)
        self.out.weight = self.emb.weight
        nn.init.normal_(self.emb.weight, 0, d ** -0.5)

    def embed(self, x):
        return self.drop(self.pos(self.emb(x) * math.sqrt(self.d)))

    def encode(self, src):
        return self.tf.encoder(self.embed(src), src_key_padding_mask=src == PAD)

    def decode(self, tgt, memory, src_pad):
        T = tgt.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt.device), 1)
        h = self.tf.decoder(self.embed(tgt), memory, tgt_mask=causal, tgt_is_causal=True,
                            tgt_key_padding_mask=tgt == PAD, memory_key_padding_mask=src_pad)
        return self.out(h)

    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src), src == PAD)


def pad_batch(seqs, value=PAD):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [value] * (L - len(s)) for s in seqs])

In [ ]:
if not done("transformer"):
    set_seed()
    sp_prefix = f"{OUT}/weights/spm_en_hi"
    with open(f"{OUT}/data/spm_corpus.txt", "w", encoding="utf-8") as f:
        for e, h in train:
            f.write(e + "\n" + h + "\n")
    spm.SentencePieceTrainer.train(
        input=f"{OUT}/data/spm_corpus.txt", model_prefix=sp_prefix, vocab_size=CFG["tf_vocab"],
        model_type="bpe", character_coverage=1.0, pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS,
        hard_vocab_limit=False, minloglevel=2)
    os.remove(f"{OUT}/data/spm_corpus.txt")
    sp = spm.SentencePieceProcessor(model_file=sp_prefix + ".model")
    ML = CFG["max_len"]

    def tf_encode(pairs):
        return [(sp.encode(e)[: ML - 1] + [EOS], [BOS] + sp.encode(h)[: ML - 2] + [EOS]) for e, h in pairs]

    tf_train, tf_valid = tf_encode(train), tf_encode(valid)

    def tf_batches(data, shuffle):
        idx = batches_by_length(len(data), lambda i: len(data[i][1]), CFG["tf_bs"], shuffle)
        return [(pad_batch([data[i][0] for i in b]), pad_batch([data[i][1] for i in b])) for b in idx]

    tf_model = Seq2SeqTransformer(len(sp), CFG["tf_d"], CFG["tf_heads"], CFG["tf_layers"],
                                  CFG["tf_ff"], CFG["tf_drop"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

    def tf_loss(model, batch):
        src, tgt = batch[0].to(DEVICE), batch[1].to(DEVICE)
        logits = model(src, tgt[:, :-1])
        return criterion(logits.float().reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))

    tf_amp = torch.bfloat16 if BF16 else torch.float16
    tf_time, tf_hist = fit(tf_model, tf_loss, tf_batches(tf_train, True), tf_batches(tf_valid, False),
                           CFG["tf_epochs"], CFG["tf_lr"], CFG["tf_warmup"], tf_amp, tag="transformer")
    torch.save(tf_model.state_dict(), f"{OUT}/weights/transformer.pt")

    @torch.no_grad()
    def tf_translate(srcs, temp=None):
        tf_model.eval()
        out = [None] * len(srcs)
        enc = [sp.encode(s)[: ML - 1] + [EOS] for s in srcs]
        for idx in batches_by_length(len(srcs), lambda i: len(enc[i]), CFG["gen_bs_seq2seq"], False):
            src = pad_batch([enc[i] for i in idx]).to(DEVICE)
            with amp_ctx(tf_amp):
                memory = tf_model.encode(src)
            ys = torch.full((len(idx), 1), BOS, device=DEVICE)
            finished = torch.zeros(len(idx), dtype=torch.bool, device=DEVICE)
            for _ in range(CFG["max_new_tokens"]):
                with amp_ctx(tf_amp):
                    logits = tf_model.decode(ys, memory, src == PAD)[:, -1].float()
                if temp:
                    nxt = torch.multinomial(torch.softmax(logits / temp, -1), 1).squeeze(1)
                else:
                    nxt = logits.argmax(-1)
                nxt = torch.where(finished, torch.full_like(nxt, PAD), nxt)
                ys = torch.cat([ys, nxt[:, None]], 1)
                finished |= nxt == EOS
                if finished.all():
                    break
            for j, i in enumerate(idx):
                toks = ys[j, 1:].tolist()
                toks = toks[: toks.index(EOS)] if EOS in toks else toks
                out[i] = sp.decode([t for t in toks if t != PAD])
        return out

    tok_stats = {"tokenizer": "SentencePiece BPE 8k (joint, trained on train split)",
                 "en_tokens_per_sent": float(np.mean([len(sp.encode(s)) for s in test_src])),
                 "hi_tokens_per_sent": float(np.mean([len(sp.encode(s)) for s in test_ref]))}
    run_system("transformer", "encoder-decoder", "nn.Transformer (scratch)", "trained from scratch",
               tf_translate, n_params(tf_model), n_params(tf_model, True), tf_time, len(train),
               tf_hist, extra={"tokenization": tok_stats})
    free(tf_model)

## 4. Model 2 – mT5-small (encoder–decoder, fine-tuned)

`google/mt5-small` (≈300 M parameters, most of them in its 250k-token multilingual
SentencePiece embedding table) was pre-trained with span corruption on mC4 in 101 languages
including Hindi, but never on translation. All weights are fine-tuned with the task prefix
`translate English to Hindi:`. mT5 is unstable in fp16, so on a T4 it trains in fp32.

In [ ]:
if not done("mt5-small"):
    set_seed()
    m_tok = AutoTokenizer.from_pretrained(CFG["mt5_id"])
    mt5 = AutoModelForSeq2SeqLM.from_pretrained(CFG["mt5_id"]).to(DEVICE)
    PREFIX = "translate English to Hindi: "
    ML = CFG["max_len"]

    def mt5_encode(pairs):
        src = m_tok([PREFIX + e for e, _ in pairs], truncation=True, max_length=ML)["input_ids"]
        tgt = m_tok([h for _, h in pairs], truncation=True, max_length=ML)["input_ids"]
        return list(zip(src, tgt))

    def mt5_batches(data, shuffle):
        idx = batches_by_length(len(data), lambda i: len(data[i][1]), CFG["mt5_bs"], shuffle)
        return [(pad_batch([data[i][0] for i in b], m_tok.pad_token_id),
                 pad_batch([data[i][1] for i in b], -100)) for b in idx]

    def mt5_loss(model, batch):
        ids = batch[0].to(DEVICE)
        return model(input_ids=ids, attention_mask=(ids != m_tok.pad_token_id).long(),
                     labels=batch[1].to(DEVICE)).loss

    mt5_amp = torch.bfloat16 if BF16 else None
    mt5_time, mt5_hist = fit(mt5, mt5_loss, mt5_batches(mt5_encode(train), True),
                             mt5_batches(mt5_encode(valid), False), CFG["mt5_epochs"],
                             CFG["mt5_lr"], CFG["mt5_warmup"], mt5_amp, accum=CFG["mt5_accum"],
                             weight_decay=0.0, tag="mt5")

    def mt5_translate(srcs, temp=None):
        return hf_translate(mt5, m_tok, [PREFIX + s for s in srcs], temp, CFG["gen_bs_seq2seq"],
                            mt5_amp, add_special=True)

    tok_stats = {"tokenizer": "mT5 SentencePiece unigram (250k, multilingual)",
                 "en_tokens_per_sent": float(np.mean([len(m_tok(s)["input_ids"]) for s in test_src])),
                 "hi_tokens_per_sent": float(np.mean([len(m_tok(s)["input_ids"]) for s in test_ref]))}
    run_system("mt5-small", "encoder-decoder", CFG["mt5_id"], "full fine-tuning", mt5_translate,
               n_params(mt5), n_params(mt5, True), mt5_time, len(train), mt5_hist,
               extra={"tokenization": tok_stats})
    free(mt5)

## 5. Decoder-only models – LLaMA 3.2 1B and Gemma 3 1B

Both are instruction-tuned decoder-only LLMs. They are adapted by **in-context prompting**;
optional LoRA fine-tuning is implemented but switched off (`llm_lora=False`) to fit the
assignment's compute budget:

* **3-shot prompting** – the base model is shown three fixed English→Hindi example pairs
  (from the training split) in its chat template and asked to translate; no weights change.
* **LoRA fine-tuning (optional, off by default)** – low-rank adapters (r = 16, α = 32) on every attention and MLP
  projection, trained for one epoch on a 10k-pair subset of the training data with the
  zero-shot instruction prompt; the loss is computed only on the Hindi answer tokens.
  Only ~1 % of the weights are trained, which is what makes 1B models tractable on a T4.

Decoding is greedy with the same `max_new_tokens` as the encoder–decoder models.
On a T4, LLaMA runs with fp16 autocast; Gemma 3 overflows in fp16, so it runs in fp32
(both use bf16 automatically on an Ampere GPU).

In [ ]:
SHOTS = [p for p in train if 6 <= len(p[0].split()) <= 12][: CFG["n_shots"]]


def chat(src, shots=()):
    msgs = []
    for e, h in shots:
        msgs += [{"role": "user", "content": LLM_INSTR.format(src=e)},
                 {"role": "assistant", "content": h}]
    return msgs + [{"role": "user", "content": LLM_INSTR.format(src=src)}]


def run_llm(key, model_id, amp_on_t4):
    """3-shot prompting, then LoRA fine-tuning, for one decoder-only model."""
    fs_name, lora_name = f"{key}-3shot", f"{key}-lora"
    if done(fs_name) and (done(lora_name) or not CFG["llm_lora"]):
        return
    set_seed()
    amp = torch.bfloat16 if BF16 else amp_on_t4
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() \
            else tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.bfloat16 if BF16 else torch.float32,
        attn_implementation="eager" if "gemma" in model_id else "sdpa").to(DEVICE)
    model.config.use_cache = True

    def render(msgs, gen_prompt=True):
        # fixed date keeps LLaMA's system header identical on every run
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen_prompt,
                                       date_string="26 Jul 2024")

    # text the chat template puts after an assistant answer (e.g. <|eot_id|> / <end_of_turn>)
    probe = render(chat("x") + [{"role": "assistant", "content": "ZZZ"}], gen_prompt=False)
    end_text = probe.split("ZZZ", 1)[1].rstrip("\n")
    stop_ids = sorted({tok.eos_token_id, *tok(end_text, add_special_tokens=False)["input_ids"][:1]})

    def translate(srcs, temp=None, shots=()):
        prompts = [render(chat(s, shots)) for s in srcs]
        return hf_translate(model, tok, prompts, temp, CFG["gen_bs_llm"], amp, add_special=False,
                            stop_ids=stop_ids, strip_prompt=True)

    tok_stats = {"tokenizer": f"{model_id} tokenizer ({len(tok)} tokens)",
                 "en_tokens_per_sent": float(np.mean([len(tok(s, add_special_tokens=False)["input_ids"]) for s in test_src])),
                 "hi_tokens_per_sent": float(np.mean([len(tok(s, add_special_tokens=False)["input_ids"]) for s in test_ref]))}
    total = n_params(model)

    if not done(fs_name):
        run_system(fs_name, "decoder-only", model_id, f"{CFG['n_shots']}-shot prompting",
                   lambda s, t: translate(s, t, SHOTS), total, 0, 0.0, 0,
                   temperature_sweep=not CFG["llm_lora"], extra={"tokenization": tok_stats})

    if CFG["llm_lora"] and not done(lora_name):
        set_seed()
        model = get_peft_model(model, LoraConfig(
            r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            task_type="CAUSAL_LM"))
        model.config.use_cache = False

        def llm_encode(pairs):
            data = []
            for e, h in pairs:
                p = tok(render(chat(e)), add_special_tokens=False)["input_ids"]
                t = tok(h + end_text, add_special_tokens=False)["input_ids"]
                ids = (p + t)[: CFG["max_llm_len"]]
                data.append((ids, ([-100] * len(p) + t)[: CFG["max_llm_len"]]))
            return data

        def llm_batches(data, shuffle):
            idx = batches_by_length(len(data), lambda i: len(data[i][0]), CFG["llm_bs"], shuffle)
            out = []
            for b in idx:
                ids = pad_batch([data[i][0] for i in b], tok.pad_token_id)
                lab = pad_batch([data[i][1] for i in b], -100)
                att = pad_batch([[1] * len(data[i][0]) for i in b], 0)
                out.append((ids, lab, att))
            return out

        def llm_loss(m, batch):
            return m(input_ids=batch[0].to(DEVICE), labels=batch[1].to(DEVICE),
                     attention_mask=batch[2].to(DEVICE)).loss

        llm_train = train[: CFG["n_llm_train"]]
        lora_time, lora_hist = fit(model, llm_loss, llm_batches(llm_encode(llm_train), True),
                                   llm_batches(llm_encode(valid), False), CFG["llm_epochs"],
                                   CFG["llm_lr"], CFG["llm_warmup"], amp, accum=CFG["llm_accum"],
                                   weight_decay=0.0, log_every=10, tag=lora_name)
        model.config.use_cache = True
        model.save_pretrained(f"{OUT}/weights/{lora_name}")
        run_system(lora_name, "decoder-only", model_id, f"LoRA r={CFG['lora_r']} fine-tuning",
                   lambda s, t: translate(s, t), total, n_params(model, True), lora_time,
                   len(llm_train), lora_hist, extra={"tokenization": tok_stats})
    free(model)

### Model 3 – LLaMA 3.2 1B Instruct (decoder-only)

In [ ]:
run_llm("llama-3.2-1b", CFG["llama_id"], amp_on_t4=torch.float16)

### Model 4 – Gemma 3 1B Instruct (decoder-only)

In [ ]:
run_llm("gemma-3-1b", CFG["gemma_id"], amp_on_t4=None)

## 6. Evaluation and comparison

BLEU (sacreBLEU, `intl` tokenisation so the Hindi danda is split off) and chrF are computed
on the same 500 test sentences for every system. chrF works on character n-grams, so it is
more forgiving than BLEU of Hindi's rich morphology and spelling variation
(e.g. हिंदी / हिन्दी, nukta and chandrabindu variants).

In [ ]:
ORDER = ["transformer", "mt5-small", "llama-3.2-1b-3shot", "llama-3.2-1b-lora",
         "gemma-3-1b-3shot", "gemma-3-1b-lora"]
LABEL = {"transformer": "Transformer (scratch)", "mt5-small": "mT5-small (FT)",
         "llama-3.2-1b-3shot": "LLaMA-3.2-1B (3-shot)", "llama-3.2-1b-lora": "LLaMA-3.2-1B (LoRA)",
         "gemma-3-1b-3shot": "Gemma-3-1B (3-shot)", "gemma-3-1b-lora": "Gemma-3-1B (LoRA)"}
R = {n: load_json(f"{OUT}/systems/{n}.json") for n in ORDER if done(n)}
COLORS = {n: ("#4C72B0" if R[n]["family"] == "encoder-decoder" else "#DD8452") for n in R}

rows = []
for n, r in R.items():
    rows.append({"System": LABEL[n], "Architecture": r["family"], "Adaptation": r["adaptation"],
                 "BLEU": r["bleu"], "chrF": r["chrf"],
                 "Params (M)": round(r["params_total"] / 1e6, 1),
                 "Trainable (M)": round(r["params_trainable"] / 1e6, 2),
                 "Train pairs": r["train_examples"],
                 "Train time (min)": round(r["train_time_s"] / 60, 1),
                 "Inference (s / 500 sent.)": r["infer_time_s"],
                 "ms / sentence": r["infer_ms_per_sent"]})
metrics = pd.DataFrame(rows)
metrics.to_csv(f"{OUT}/metrics.csv", index=False)
score(test_ref[:2], test_ref[:2])   # sacreBLEU only exposes its signature after one evaluation
with open(f"{OUT}/metrics.md", "w", encoding="utf-8") as f:
    f.write(metrics.to_markdown(index=False) + f"\n\nBLEU: `{BLEU_METRIC.get_signature()}`  \n"
            f"chrF: `{CHRF_METRIC.get_signature()}`  \nGPU: {GPU_NAME}\n")
print(metrics.to_string(index=False))

tok_rows = [{"System": LABEL[n], **R[n]["tokenization"]} for n in R if "tokenization" in R[n]]
tok_df = pd.DataFrame(tok_rows).drop_duplicates("tokenizer")
tok_df["hi/en token ratio"] = (tok_df.hi_tokens_per_sent / tok_df.en_tokens_per_sent).round(2)
tok_df.to_csv(f"{OUT}/tokenization.csv", index=False)
print(tok_df.to_string(index=False))

In [ ]:
names = list(R)
x = np.arange(len(names))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, m in zip(ax, ["bleu", "chrf"]):
    v = [R[n][m] for n in names]
    a.bar(x, v, color=[COLORS[n] for n in names])
    for i, val in enumerate(v):
        a.text(i, val, f"{val:.1f}", ha="center", va="bottom", fontsize=8)
    a.set_xticks(x, [LABEL[n] for n in names], rotation=30, ha="right", fontsize=8)
    a.set_title(m.upper() if m == "bleu" else "chrF")
fig.suptitle(f"Translation quality on the shared {len(test_src)}-sentence test set "
             "(blue = encoder–decoder, orange = decoder-only)", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/plots/quality.png"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for n in names:
    ax[0].scatter(R[n]["params_total"] / 1e6, R[n]["chrf"], color=COLORS[n], s=50)
    ax[0].annotate(LABEL[n], (R[n]["params_total"] / 1e6, R[n]["chrf"]), fontsize=7)
ax[0].set(xscale="log", xlabel="total parameters (M, log)", ylabel="chrF", title="Size vs quality")
ax[1].bar(x, [R[n]["infer_ms_per_sent"] for n in names], color=[COLORS[n] for n in names])
ax[1].set_xticks(x, [LABEL[n] for n in names], rotation=30, ha="right", fontsize=8)
ax[1].set(title="Inference time (ms / sentence, greedy)")
ax[2].bar(x, [R[n]["train_time_s"] / 60 for n in names], color=[COLORS[n] for n in names])
ax[2].set_xticks(x, [LABEL[n] for n in names], rotation=30, ha="right", fontsize=8)
ax[2].set(title="Training / fine-tuning time (min)")
plt.tight_layout(); plt.savefig(f"{OUT}/plots/efficiency.png"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for n in names:
    h = R[n]["history"]
    if h.get("train_loss"):
        s = np.array(h["step"], dtype=float)
        ax[0].plot(s / s.max(), h["train_loss"], label=LABEL[n])
        ax[1].plot(h["epoch"], h["val_loss"], marker="o", label=LABEL[n])
ax[0].set(title="Training loss", xlabel="fraction of training", ylabel="loss"); ax[0].legend(fontsize=7)
ax[1].set(title="Validation loss per epoch", xlabel="epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/plots/training_curves.png"); plt.show()

### Sampling-temperature analysis
For the four systems (Transformer, mT5, LLaMA 3-shot, Gemma 3-shot) the first 100 test
sentences are decoded with pure temperature sampling (`top_k = 0`, `top_p = 1.0`)
at T ∈ {0.3, 0.7, 1.0, 1.5}; T = 0 is greedy decoding.

In [ ]:
temp_rows = []
for n in names:
    for t, v in R[n].get("temperature", {}).items():
        temp_rows.append({"System": LABEL[n], "T": float(t), "BLEU": v["bleu"], "chrF": v["chrf"]})
temp_df = pd.DataFrame(temp_rows)
temp_df.to_csv(f"{OUT}/temperature.csv", index=False)
if len(temp_df):
    print(temp_df.pivot(index="T", columns="System", values="chrF").to_string())
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for s, g in temp_df.groupby("System", sort=False):
        ax[0].plot(g["T"], g["BLEU"], marker="o", label=s)
        ax[1].plot(g["T"], g["chrF"], marker="o", label=s)
    ax[0].set(title="BLEU vs sampling temperature", xlabel="temperature (0 = greedy)")
    ax[1].set(title="chrF vs sampling temperature", xlabel="temperature (0 = greedy)")
    ax[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{OUT}/plots/temperature.png"); plt.show()

with open(f"{OUT}/temperature_examples.md", "w", encoding="utf-8") as f:
    for k in range(min(3, CFG["n_temp"])):
        f.write(f"### Example {k + 1}\n\n**EN:** {test_src[k]}  \n**REF:** {test_ref[k]}\n\n"
                "| System | T | Output |\n|---|---|---|\n")
        for n in names:
            for t, v in R[n].get("temperature", {}).items():
                f.write(f"| {LABEL[n]} | {t} | {v['hyps'][k]} |\n")
        f.write("\n")

## 7. Responsible-AI checks

Automatic, reference-free risk indicators on the 500 test outputs:
* **number mismatch** – a number in the English source is missing or a new number appears
  (Devanagari digits are mapped to ASCII first) → hallucinated / dropped facts;
* **untranslated** – more than 30 % of the letters in the output are Latin;
* **over-generation** – output more than 2× the reference length (added content);
* **under-generation** – output less than 0.4× the reference length (dropped content);
* **repetition** – some 3-gram occurs 3 or more times (degenerate looping);
* **empty** output.

The probe-set translations (gender, facts, idioms, culture, sensitive and safety-critical
sentences) are written to `probe_translations.md` for manual inspection.

In [ ]:
DEVA_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
NUM = re.compile(r"\d+(?:[.,]\d+)?")


def risk_flags(src, hyp, ref):
    hs, rs = hyp.split(), ref.split()
    s_nums = Counter(n.replace(",", "") for n in NUM.findall(src))
    h_nums = Counter(n.replace(",", "") for n in NUM.findall(hyp.translate(DEVA_DIGITS)))
    deva, lat = len(DEVA.findall(hyp)), len(LAT.findall(hyp))
    grams = Counter(tuple(hs[i:i + 3]) for i in range(len(hs) - 2))
    return {"number_mismatch": bool(s_nums or h_nums) and s_nums != h_nums,
            "untranslated": lat > 0.3 * max(1, deva + lat),
            "over_generation": len(hs) > 2 * len(rs),
            "under_generation": len(hs) < 0.4 * len(rs),
            "repetition": bool(grams) and max(grams.values()) >= 3,
            "empty": len(hs) == 0}


risk_rows = []
for n in names:
    flags = [risk_flags(s, h, r) for s, h, r in zip(test_src, R[n]["hyps"], test_ref)]
    risk_rows.append({"System": LABEL[n], **{k: round(100 * np.mean([f[k] for f in flags]), 1)
                                              for k in flags[0]}})
risk = pd.DataFrame(risk_rows)
risk.to_csv(f"{OUT}/risk_flags.csv", index=False)
print("% of test outputs flagged")
print(risk.to_string(index=False))

ax = risk.set_index("System").drop(columns="empty").plot.bar(figsize=(11, 3.8), rot=25)
ax.set(ylabel="% of test outputs", title="Automatic risk indicators (lower is better)")
plt.tight_layout(); plt.savefig(f"{OUT}/plots/risk_flags.png"); plt.show()

with open(f"{OUT}/probe_translations.md", "w", encoding="utf-8") as f:
    for k, (cat, s) in enumerate(PROBES):
        f.write(f"### {k + 1}. [{cat}] {s}\n\n| System | Translation |\n|---|---|\n")
        for n in names:
            f.write(f"| {LABEL[n]} | {R[n]['probe'][k]} |\n")
        f.write("\n")

with open(f"{OUT}/sample_translations.md", "w", encoding="utf-8") as f:
    for k in range(min(15, len(test_src))):
        f.write(f"### Test sentence {k + 1}\n\n**EN:** {test_src[k]}  \n**REF:** {test_ref[k]}\n\n"
                "| System | Translation |\n|---|---|\n")
        for n in names:
            f.write(f"| {LABEL[n]} | {R[n]['hyps'][k]} |\n")
        f.write("\n")

all_tr = pd.DataFrame({"en": test_src, "reference": test_ref, **{n: R[n]["hyps"] for n in names}})
all_tr.to_csv(f"{OUT}/test_translations.tsv", sep="\t", index=False)
print(open(f"{OUT}/probe_translations.md", encoding="utf-8").read()[:3000])

## 8. Package the results
Everything except model weights is zipped (`results.zip`) and downloaded; this is the
`results/` folder of the GitHub repository.

In [ ]:
save_json({k: v for k, v in CFG.items()}, f"{OUT}/config.json")
pkg = "/content/results_pkg" if IN_COLAB else f"{OUT}_pkg"
shutil.rmtree(pkg, ignore_errors=True)
shutil.copytree(OUT, pkg, ignore=shutil.ignore_patterns("weights", "spm_corpus.txt"))
shutil.make_archive("results", "zip", pkg)
print("wrote results.zip")
if IN_COLAB:
    from google.colab import files
    files.download("results.zip")